# 保险保费预测 - 模型训练

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import torch
import optuna
import warnings
import faiss
import random

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from category_encoders import TargetEncoder
from sklearn.feature_selection import mutual_info_regression, VarianceThreshold
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge

warnings.filterwarnings('ignore')
SEED = 42

def seed_everything(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    random.seed(seed)

seed_everything()

def check_lgb_gpu():
    try:
        lgb.cuda_is_available()
        print('✅ LightGBM GPU')
        return True
    except:
        print('⚠️ CPU mode')
        return False

def faiss_knn_imputer_gpu(X, n_neighbors=5):
    X = X.astype(np.float32)
    non_nan_mask = ~np.isnan(X).any(axis=1)
    X_non_nan = X[non_nan_mask]
    index = faiss.IndexFlatL2(X.shape[1])
    index.add(X_non_nan)
    X_imputed = X.copy()
    nan_rows = np.where(np.isnan(X).any(axis=1))[0]
    for i in nan_rows:
        query = np.nan_to_num(X[i]).reshape(1,-1)
        D, I = index.search(query, n_neighbors)
        neighbors = X[non_nan_mask][I[0]]
        X_imputed[i] = np.nanmean(neighbors, axis=0)
    return X_imputed

def rmsle(y_true, y_pred):
    y_true = np.clip(y_true, 1e-6, None)
    y_pred = np.clip(y_pred, 1e-6, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

def get_dynamic_weights(scores):
    scores = np.array(scores)
    inv = 1 / (scores + 1e-8)
    return inv / inv.sum()

In [ ]:
def feature_engineering(train_path, test_path):
    train_pl = pl.read_csv(train_path)
    test_pl = pl.read_csv(test_path)
    target = 'Premium Amount'
    y = train_pl.select(target).to_pandas().values.ravel()

    train_pl = train_pl.rename({c: c.strip().lower().replace(' ', '_') for c in train_pl.columns})
    test_pl = test_pl.rename({c: c.strip().lower().replace(' ', '_') for c in test_pl.columns})
    target = target.lower().replace(' ', '_')

    train_feats = train_pl.drop(target)
    test_feats = test_pl
    df = pl.concat([train_feats, test_feats], how='vertical')

    for col in df.columns:
        if df[col].dtype in [pl.String, pl.Boolean]:
            df = df.with_columns(pl.col(col).fill_null('unknown'))
        else:
            if col != 'id':
                df = df.with_columns(pl.col(col).fill_null(pl.col(col).median()).fill_null(0))

    df = df.with_columns([
        (pl.col('age') * pl.col('vehicle_age')).alias('age_vehicleage'),
        (pl.col('annual_income') * pl.col('health_score')).alias('income_health'),
        (pl.col('credit_score') * pl.col('insurance_duration')).alias('credit_duration'),
        (pl.col('age') * pl.col('credit_score')).alias('age_credit'),
        (pl.col('health_score') / (pl.col('age') + 1)).alias('health_per_age'),
        (pl.col('vehicle_age') / (pl.col('age') + 1)).alias('vehicle_per_age'),
        (pl.col('credit_score') / (pl.col('annual_income') + 1e-8)).alias('credit_per_income'),
        (pl.col('insurance_duration') / (pl.col('vehicle_age') + 1e-8)).alias('duration_per_vehicle'),
        (pl.col('annual_income') / (pl.col('age') + 1)).alias('income_per_age'),
        (pl.col('credit_score') / (pl.col('health_score') + 1e-8)).alias('credit_per_health'),
        (pl.col('insurance_duration') / (pl.col('age') + 1)).alias('duration_per_age'),
        (pl.col('annual_income').log1p()).alias('log_income'),
        (pl.col('age') * pl.col('health_score')).alias('age_health'),
    ])

    for c in ['location', 'occupation', 'marital_status', 'education', 'policy_type']:
        if c in df.columns:
            cnt = df.group_by(c).len().to_dicts()
            cnt_map = {d[c]: d['len'] for d in cnt}
            df = df.with_columns(pl.col(c).replace(cnt_map, default=0).alias(f'{c}_cnt'))

    if 'policy_start_date' in df.columns:
        df = df.with_columns(pl.col('policy_start_date').str.to_datetime(strict=False))
        df = df.with_columns([
            pl.col('policy_start_date').dt.year().fill_null(0).alias('policy_year'),
            pl.col('policy_start_date').dt.month().fill_null(0).alias('policy_month'),
            pl.col('policy_start_date').dt.quarter().fill_null(0).alias('policy_quarter'),
        ])

    for c in df.columns:
        if df[c].dtype == pl.String:
            df = df.with_columns(pl.col(c).cast(pl.Categorical).to_physical())

    X_train = df.head(len(train_feats)).to_pandas()
    X_test = df.tail(len(test_feats)).to_pandas()

    cat_feats = [c for c in X_train.columns if X_train[c].nunique() < 50 and c != 'id']
    te = TargetEncoder(cols=cat_feats, smoothing=10)
    X_train = pd.concat([X_train, te.fit_transform(X_train[cat_feats], y).add_prefix('te_')], axis=1)
    X_test = pd.concat([X_test, te.transform(X_test[cat_feats]).add_prefix('te_')], axis=1)

    drop_cols = ['id', 'policy_start_date']
    X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns], errors='ignore')
    X_test = X_test.drop(columns=[c for c in drop_cols if c in X_test.columns], errors='ignore')
    X_train = X_train.fillna(0).astype(float)
    X_test = X_test.fillna(0).astype(float)

    vt = VarianceThreshold(0.01)
    X_train = vt.fit_transform(X_train)
    X_test = vt.transform(X_test)

    mi = mutual_info_regression(X_train, y, random_state=SEED, n_jobs=-1)
    mask = mi >= np.percentile(mi, 10)
    X_train = X_train[:, mask]
    X_test = X_test[:, mask]

    return X_train, X_test, y

In [ ]:
def objective(trial, X, y_log, lgb_gpu):
    kf = KFold(5, shuffle=True, random_state=SEED)
    params = {
        'lgb': {
            'learning_rate': trial.suggest_float('lgb_learning_rate', 0.05, 0.15),
            'max_depth': trial.suggest_int('lgb_max_depth', 3,6),
            'reg_alpha': trial.suggest_float('lgb_reg_alpha', 0.01,2),
            'reg_lambda': trial.suggest_float('lgb_reg_lambda', 0.01,2),
            'colsample_bytree':0.7, 'subsample':0.7, 'n_estimators':300,
            'random_state':SEED, 'verbose':-1
        },
        'xgb': {
            'learning_rate': trial.suggest_float('xgb_learning_rate', 0.05,0.15),
            'max_depth': trial.suggest_int('xgb_max_depth',3,6),
            'reg_alpha':0.1, 'reg_lambda':1.0, 'colsample_bytree':0.7, 'subsample':0.7,
            'eval_metric':'rmse', 'n_estimators':300, 'random_state':SEED
        },
        'cat': {
            'learning_rate': trial.suggest_float('cat_learning_rate',0.05,0.12),
            'depth': trial.suggest_int('cat_depth',3,5),
            'l2_leaf_reg': trial.suggest_float('cat_l2_leaf_reg',1,5),
            'n_estimators':300, 'random_state':SEED
        }
    }

    scores, model_scores = [], []
    for tr, val in kf.split(X):
        xt, xv = X[tr], X[val]
        yt, yv = y_log[tr], y_log[val]
        scaler = StandardScaler()
        xs_t = scaler.fit_transform(xt)
        xs_v = scaler.transform(xv)

        m1 = lgb.LGBMRegressor(**params['lgb'], device='cuda', gpu_use_dp=False)
        m1.fit(xt, yt, eval_set=[(xv,yv)], callbacks=[lgb.early_stopping(50, verbose=0)])

        m2 = xgb.XGBRegressor(**params['xgb'], tree_method='hist', device='cuda')
        m2.fit(xt, yt, eval_set=[(xv,yv)], verbose=False)

        m3 = CatBoostRegressor(**params['cat'], task_type='GPU', verbose=0)
        m3.fit(xt, yt, eval_set=(xv,yv), use_best_model=True)

        m4 = lgb.LGBMRegressor(boosting_type='rf', n_estimators=100, max_depth=6,
                               bagging_freq=1, bagging_fraction=0.8, feature_fraction=0.8,
                               device='cuda', verbose=-1)
        m4.fit(xt, yt)

        m5 = ExtraTreesRegressor(n_estimators=100, max_depth=6, n_jobs=-1, random_state=SEED)
        m5.fit(xs_t, yt)

        m6 = Ridge(alpha=1.0, random_state=SEED)
        m6.fit(xs_t, yt)

        p1,p2,p3,p4,p5,p6 = m1.predict(xv),m2.predict(xv),m3.predict(xv),m4.predict(xv),m5.predict(xs_v),m6.predict(xs_v)
        yv_real = np.expm1(yv)
        s = [rmsle(yv_real, np.expm1(p)) for p in [p1,p2,p3,p4,p5,p6]]
        model_scores.append(s)
        w = get_dynamic_weights(s)
        final = np.average([p1,p2,p3,p4,p5,p6], weights=w, axis=0)
        scores.append(rmsle(yv_real, np.expm1(final)))

    return np.mean(scores)

In [ ]:
def train_stacking(X, y_log, X_test, best_params, lgb_gpu):
    kf = KFold(10, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X),6))
    pred = np.zeros((len(X_test),6))
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    Xt_s = scaler.transform(X_test)
    model_scores = []

    for fold, (tr,val) in enumerate(kf.split(X)):
        print(f'Fold {fold+1}/10')
        xt,xv = X[tr],X[val]
        yt,yv = y_log[tr],y_log[val]
        xs_t,xs_v = X_s[tr],X_s[val]

        m1 = lgb.LGBMRegressor(**best_params['lgb'])
        m1.fit(xt,yt,eval_set=[(xv,yv)], callbacks=[lgb.early_stopping(200, verbose=0)])

        m2 = xgb.XGBRegressor(**best_params['xgb'])
        m2.fit(xt,yt,eval_set=[(xv,yv)], verbose=False)

        m3 = CatBoostRegressor(**best_params['cat'])
        m3.fit(xt,yt,eval_set=(xv,yv), use_best_model=True)

        m4 = lgb.LGBMRegressor(boosting_type='rf', bagging_freq=1, bagging_fraction=0.8, device='cuda', verbose=-1)
        m4.fit(xt,yt)

        m5 = ExtraTreesRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=SEED)
        m5.fit(xs_t,yt)

        m6 = Ridge(alpha=1.0, random_state=SEED)
        m6.fit(xs_t,yt)

        oof[val] = np.column_stack([m1.predict(xv),m2.predict(xv),m3.predict(xv),m4.predict(xv),m5.predict(xs_v),m6.predict(xs_v)])
        pred[:,0] += m1.predict(X_test)/10
        pred[:,1] += m2.predict(X_test)/10
        pred[:,2] += m3.predict(X_test)/10
        pred[:,3] += m4.predict(X_test)/10
        pred[:,4] += m5.predict(Xt_s)/10
        pred[:,5] += m6.predict(Xt_s)/10

        yv_real = np.expm1(yv)
        s = [rmsle(yv_real, np.expm1(oof[val][:,i])) for i in range(6)]
        model_scores.append(s)

    w = get_dynamic_weights(np.mean(model_scores, axis=0))
    print('动态权重:', np.round(w,4))
    return pred @ w

In [ ]:
# ========== 主运行 ==========
TRAIN = 'data/train.csv'
TEST = 'data/test.csv'

X, Xt, y = feature_engineering(TRAIN, TEST)
lgb_gpu = check_lgb_gpu()

X = faiss_knn_imputer_gpu(X,5)
Xt = faiss_knn_imputer_gpu(Xt,5)

y = np.clip(y, *np.percentile(y, [1,99]))
y_log = np.log1p(y)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(lambda trial: objective(trial,X,y_log,lgb_gpu), n_trials=10)

best = study.best_params
best_params = {
    'lgb': {
        'learning_rate': best['lgb_learning_rate'], 'max_depth': best['lgb_max_depth'],
        'reg_alpha': best['lgb_reg_alpha'], 'reg_lambda': best['lgb_reg_lambda'],
        'colsample_bytree':0.7, 'subsample':0.7, 'device':'cuda', 'gpu_use_dp':False,
        'n_estimators':2000, 'random_state':SEED, 'verbose':-1
    },
    'xgb': {
        'learning_rate': best['xgb_learning_rate'], 'max_depth': best['xgb_max_depth'],
        'reg_alpha':0.1, 'reg_lambda':1.0, 'colsample_bytree':0.7, 'subsample':0.7,
        'tree_method':'hist', 'device':'cuda', 'eval_metric':'rmse', 'random_state':SEED
    },
    'cat': {
        'learning_rate': best['cat_learning_rate'], 'depth': best['cat_depth'],
        'l2_leaf_reg': best['cat_l2_leaf_reg'], 'task_type':'GPU', 'random_state':SEED, 'verbose':0
    }
}

final_log = train_stacking(X,y_log,Xt,best_params,lgb_gpu)
final_pred = np.expm1(final_log)*0.99
final_pred = np.clip(final_pred, *np.percentile(y,[1,99]))

sub = pd.DataFrame({
    'id': pd.read_csv(TEST)['id'],
    'Premium Amount': final_pred.round(3)
})
sub.to_csv('submission.csv', index=False)
print('✅ 完成')